# DPS inverse experiments on one validation shard

This notebook clones the repo, downloads only one validation shard and the EMA-only checkpoint from Yandex Disk, wraps the EMA weights into the checkpoint format expected by the shared inverse pipeline, creates shared case files, and runs DPS with and without the physics-informed divergence loss.

Important reproducibility convention: `METRIC_SAMPLE_COUNT` controls how many cases are used for metrics. The first `VISUALIZATION_COUNT` cases are selected with `VISUALIZATION_SEED` and placed first, so the PNG visualizations stay identical when different people use different metric sample counts.

In [ ]:
#@title 1. Clone repo and install dependencies
from pathlib import Path
import os

REPO_URL = "https://github.com/SadreevAmir/DL_final_project.git"  #@param {type:"string"}
REPO_DIR = Path("/content/DL_final_project")

if not REPO_DIR.exists():
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    %cd "$REPO_DIR"
    !git pull --ff-only

%cd "$REPO_DIR"
!pip install -q -r requirements.txt


In [ ]:
#@title 2. Experiment settings
from pathlib import Path

DATA_SOURCE = "https://disk.yandex.ru/d/rrjDGzzX5cfFnA"  #@param {type:"string"}
EMA_CHECKPOINT_NAME = "best_score_vp_kolmogorov_velocity_256_to_64_coords_2ch_64x64_coords_bf16_20260517_221201_epoch_0014_val_0.028523_ema_model_only.pt"  #@param {type:"string"}
VAL_SHARD_NAME = "val_000.npz"  #@param {type:"string"}

# Operators can be any subset of: sparse_grid, center_box, downsample, blur.
OPERATORS = ["sparse_grid", "center_box", "downsample", "blur"]

# Metrics use this many validation samples from the downloaded val shard.
METRIC_SAMPLE_COUNT = 16  #@param {type:"integer"}
SAMPLE_SEED = 0  #@param {type:"integer"}

# Keep these fixed across people/runs to make visualizations identical.
VISUALIZATION_COUNT = 8  #@param {type:"integer"}
VISUALIZATION_SEED = 0  #@param {type:"integer"}
MAX_VISUALIZATIONS = VISUALIZATION_COUNT

# Keep this fixed for the same corruption noise.
CORRUPTION_SEED = 0  #@param {type:"integer"}
NOISE_SIGMA = 0.0  #@param {type:"number"}

# DPS settings. Start with 64/128 for debug, then increase to 256 for final runs.
DPS_STEPS = 128  #@param {type:"integer"}
BATCH_SIZE = 4  #@param {type:"integer"}
DPS_SEED = 0  #@param {type:"integer"}
GUIDANCE_SCALE = 1.0  #@param {type:"number"}
MEASUREMENT_SIGMA = 0.0  #@param {type:"number"}
GUIDANCE_START = 1.0  #@param {type:"number"}
GUIDANCE_END = 0.0  #@param {type:"number"}
GRADIENT_CLIP = 1.0  #@param {type:"number"}

# The notebook runs regular DPS and DPS + physics for every value here.
# 0.0 means no physics-informed loss.
DIV_WEIGHTS = [0.0, 0.01]

CACHE_DIR = Path("data/one_val_cache")
CHECKPOINT_DIR = Path("checkpoints")
CASE_DIR = Path("inverse_cases")
RUNS_DIR = Path("runs_inverse_colab")
WRAPPED_CHECKPOINT_PATH = CHECKPOINT_DIR / "wrapped_ema_checkpoint.pt"

# EMA-only checkpoints do not contain mean/std. If you have exact train stats,
# paste them here. If left as None, the notebook computes shard stats from val_000.
TRAIN_MEAN = None
TRAIN_STD = None


In [ ]:
#@title 3. Imports and device
import json
import math
import urllib.parse
import urllib.request
from collections import deque

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

from score_training import DiffusersUNet, VPCosineSDE
from inverse.cases import CaseConfig, create_case_file
from inverse.experiment import ExperimentConfig, run_experiment

torch.manual_seed(DPS_SEED)
np.random.seed(DPS_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


In [ ]:
#@title 4. Download only one val shard and the EMA checkpoint
def yandex_json(url: str) -> dict:
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response:
        return json.loads(response.read().decode("utf-8"))


def yandex_list(public_key: str, path: str = "/") -> dict:
    params = {"public_key": public_key, "path": path, "limit": 1000}
    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources?" + urllib.parse.urlencode(params)
    return yandex_json(api_url)


def find_public_path(public_key: str, filename: str) -> str:
    queue = deque(["/"])
    seen = set()
    while queue:
        path = queue.popleft()
        if path in seen:
            continue
        seen.add(path)
        payload = yandex_list(public_key, path)
        items = payload.get("_embedded", {}).get("items", [])
        for item in items:
            name = item.get("name", "")
            item_path = item.get("path", "")
            public_path = item_path.split(":", 1)[-1] if ":" in item_path else item_path
            if name == filename:
                return public_path
            if item.get("type") == "dir":
                queue.append(public_path)
    raise FileNotFoundError(f"Could not find {filename!r} in Yandex public folder")


def yandex_download_href(public_key: str, path: str) -> str:
    params = {"public_key": public_key, "path": path}
    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download?" + urllib.parse.urlencode(params)
    return yandex_json(api_url)["href"]


def download_file(url: str, target: Path, desc: str) -> Path:
    target.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response:
        total = int(response.headers.get("Content-Length", 0))
        if target.exists() and (total <= 0 or target.stat().st_size == total):
            print("using cached:", target)
            return target
        with target.open("wb") as handle, tqdm(total=total or None, unit="B", unit_scale=True, desc=desc) as bar:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                handle.write(chunk)
                bar.update(len(chunk))
    return target


val_public_path = find_public_path(DATA_SOURCE, VAL_SHARD_NAME)
ckpt_public_path = find_public_path(DATA_SOURCE, EMA_CHECKPOINT_NAME)
print("val path:", val_public_path)
print("checkpoint path:", ckpt_public_path)

val_path = CACHE_DIR / VAL_SHARD_NAME
checkpoint_path = CHECKPOINT_DIR / EMA_CHECKPOINT_NAME
download_file(yandex_download_href(DATA_SOURCE, val_public_path), val_path, VAL_SHARD_NAME)
download_file(yandex_download_href(DATA_SOURCE, ckpt_public_path), checkpoint_path, EMA_CHECKPOINT_NAME)
print("val shard:", val_path, f"{val_path.stat().st_size / 1024**2:.1f} MB")
print("checkpoint:", checkpoint_path, f"{checkpoint_path.stat().st_size / 1024**2:.1f} MB")


In [ ]:
#@title 5. Inspect val shard and prepare mean/std
with np.load(val_path) as arrays:
    print("keys:", list(arrays.keys()))
    val_images = arrays["images"].astype(np.float32, copy=False)
print("val images:", val_images.shape, val_images.dtype)

if TRAIN_MEAN is None or TRAIN_STD is None:
    print("WARNING: TRAIN_MEAN/TRAIN_STD are not set. Using stats computed from the single val shard.")
    print("For final numbers, paste exact train-set stats if available.")
    mean = val_images.mean(axis=(0, 2, 3), dtype=np.float64).astype(np.float32)
    std = val_images.std(axis=(0, 2, 3), dtype=np.float64).astype(np.float32)
else:
    mean = np.asarray(TRAIN_MEAN, dtype=np.float32)
    std = np.asarray(TRAIN_STD, dtype=np.float32)
std = np.maximum(std, 1.0e-6)
print("mean:", mean)
print("std:", std)


In [ ]:
#@title 6. Wrap EMA-only checkpoint into the shared pipeline checkpoint format
CHANNELS = 2
IMAGE_SIZE = 64
COORDINATE_MODE = "fourier"
CHANNELS_PER_LEVEL = "96,192,384"
NUM_RES_BLOCKS = 3
ATTENTION_HEAD_DIM = 32
PADDING_MODE = "circular"
TIME_EMBEDDING_SCALE = 999.0
CLIP_PRED_X0 = 5.0

state = torch.load(checkpoint_path, map_location="cpu")
if isinstance(state, dict) and "ema_model" in state:
    state = state["ema_model"]
config = {
    "channels_per_level": CHANNELS_PER_LEVEL,
    "num_res_blocks": NUM_RES_BLOCKS,
    "attention_head_dim": ATTENTION_HEAD_DIM,
    "padding_mode": PADDING_MODE,
    "coordinate_mode": COORDINATE_MODE,
    "time_embedding_scale": TIME_EMBEDDING_SCALE,
    "clip_pred_x0": CLIP_PRED_X0,
    "sample_steps": DPS_STEPS,
    "dropout": 0.0,
}
data_stats = {
    "mean": [float(x) for x in mean],
    "std": [float(x) for x in std],
    "channels": CHANNELS,
    "height": IMAGE_SIZE,
    "width": IMAGE_SIZE,
    "train_count": 0,
    "val_count": int(val_images.shape[0]),
    "source": str(val_path),
}
wrapped = {
    "ema_model": state,
    "config": config,
    "data_stats": data_stats,
    "coordinate_mode": COORDINATE_MODE,
    "time_embedding_scale": TIME_EMBEDDING_SCALE,
}
WRAPPED_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(wrapped, WRAPPED_CHECKPOINT_PATH)
print("wrapped checkpoint:", WRAPPED_CHECKPOINT_PATH)


In [ ]:
#@title 7. Create shared case files
case_paths = {}
for operator_name in OPERATORS:
    case_path = CASE_DIR / f"{operator_name}_val{METRIC_SAMPLE_COUNT}_vis{VISUALIZATION_COUNT}_seed{SAMPLE_SEED}.npz"
    cfg = CaseConfig(
        data_source=str(CACHE_DIR),
        output_path=str(case_path),
        operator=operator_name,
        split="val",
        num_samples=METRIC_SAMPLE_COUNT,
        sample_seed=SAMPLE_SEED,
        visualization_count=VISUALIZATION_COUNT,
        visualization_seed=VISUALIZATION_SEED,
        corruption_seed=CORRUPTION_SEED,
        noise_sigma=NOISE_SIGMA,
        device="cpu",
    )
    case_paths[operator_name] = create_case_file(cfg)
    print(operator_name, "->", case_paths[operator_name])


In [ ]:
#@title 8. Run DPS experiments
metrics_paths = []
for operator_name, case_path in case_paths.items():
    for div_weight in DIV_WEIGHTS:
        tag = "dps" if div_weight == 0.0 else f"dps_div{div_weight:g}".replace(".", "p")
        output_dir = RUNS_DIR / f"{tag}_{operator_name}_n{METRIC_SAMPLE_COUNT}_steps{DPS_STEPS}"
        cfg = ExperimentConfig(
            checkpoint_path=str(WRAPPED_CHECKPOINT_PATH),
            case_file=str(case_path),
            output_dir=str(output_dir),
            method="dps",
            device="auto",
            batch_size=BATCH_SIZE,
            steps=DPS_STEPS,
            seed=DPS_SEED,
            max_visualizations=MAX_VISUALIZATIONS,
            guidance_scale=GUIDANCE_SCALE,
            measurement_sigma=MEASUREMENT_SIGMA,
            guidance_start=GUIDANCE_START,
            guidance_end=GUIDANCE_END,
            gradient_clip=GRADIENT_CLIP,
            div_weight=float(div_weight),
        )
        print("running:", output_dir)
        metrics_paths.append(run_experiment(cfg))
        print("saved:", metrics_paths[-1])

metrics_paths


In [ ]:
#@title 9. Summarize metrics
import pandas as pd

frames = []
for path in metrics_paths:
    df = pd.read_csv(path)
    df["run"] = str(path.parent)
    frames.append(df)
all_metrics = pd.concat(frames, ignore_index=True)
summary = all_metrics.groupby(["method", "operator", "run"])[[
    "rel_l2", "rmse", "measurement_error", "divergence", "vorticity_rmse"
]].mean().reset_index()
display(summary)
summary.to_csv(RUNS_DIR / "summary.csv", index=False)
print("summary:", RUNS_DIR / "summary.csv")


## How to keep visualizations identical

- Change `METRIC_SAMPLE_COUNT` to use more or fewer samples for metrics.
- Keep `VISUALIZATION_COUNT` and `VISUALIZATION_SEED` fixed across runs and across people.
- Keep `MAX_VISUALIZATIONS <= VISUALIZATION_COUNT`.
- Keep `CORRUPTION_SEED` fixed if you want noisy corruptions to be identical.

The shared case generator puts the visualization samples first in the case file. The experiment runner always visualizes the first `MAX_VISUALIZATIONS` samples, so those images stay fixed even when the total metric sample count changes.